# FloodNet U-Net training (canonical project workflow)

This notebook calls the maintained `src.train` and `src.evaluate` modules. It does not keep a second training implementation. Training uses the FloodNet train and validation splits; the held-out test split is reserved for final evaluation. Checkpoints and history are mirrored to Google Drive.

The existing handover checkpoint had zero Road-flooded validation IoU, so do not present this model as a reliable waterlogging detector until Road-flooded performance improves.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Get the project and dataset ready

Place an extracted FloodNet Supervised v1.0 directory at `MyDrive/FloodNet-Supervised_v1.0/` so it contains `train/`, `val/`, and `test/`. Dataset archives and images stay outside GitHub. The commands below clone this existing repository only when it is not already present in the Colab runtime.

In [ ]:
from pathlib import Path
import os, subprocess, sys

PROJECT_DIR = Path("/content/Road_Waterlogging_Project-")
if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/vaishnavij1612/Road_WaterLogging_Project-.git", str(PROJECT_DIR)], check=True)
os.chdir(PROJECT_DIR)
FLOODNET_DIR = Path("/content/drive/MyDrive/FloodNet-Supervised_v1.0")
assert (FLOODNET_DIR / "train/train-org-img").is_dir(), f"FloodNet not found at {FLOODNET_DIR}"
assert (FLOODNET_DIR / "val/val-label-img").is_dir(), f"FloodNet masks not found at {FLOODNET_DIR}"
DRIVE_CHECKPOINTS = Path("/content/drive/MyDrive/FloodNet_Checkpoints")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Project:", PROJECT_DIR, "| Dataset:", FLOODNET_DIR, "| GPU:", subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0)

## Train or resume

`--epochs` is the total target epoch. The maintained script resumes the latest valid checkpoint, saves a latest checkpoint every completed epoch, and saves the best checkpoint by validation mIoU.

In [ ]:
subprocess.run([
    sys.executable, "-m", "src.train",
    "--floodnet-dir", str(FLOODNET_DIR),
    "--epochs", "15", "--batch-size", "4",
    "--drive-dir", str(DRIVE_CHECKPOINTS),
], check=True)

## Validate and inspect real predictions

Use validation results for model selection. The evaluator writes JSON metrics and genuine original/ground-truth/prediction comparison images.

In [ ]:
subprocess.run([sys.executable, "-m", "src.evaluate", "--split", "val"], check=True)

## Final test evaluation

Run this once after selecting the model; do not use test results for further tuning.

In [ ]:
subprocess.run([sys.executable, "-m", "src.evaluate", "--split", "test"], check=True)